In [14]:
# Import necessary libraries
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Load the dataset
df = pd.read_csv('../data/combined_global_gender_data_20251123.csv')
print('Dataset shape:', df.shape)
print('Columns:', list(df.columns))
df.head()

Dataset shape: (193, 6)
Columns: ['Scraped_Date', 'Data_Year', 'Country', 'ISO3', 'Year', 'Gender Inequality Index (GII) (Gender Inequality Index)']


,Scraped_Date,Data_Year,Country,ISO3,Year,Gender Inequality Index (GII) (Gender Inequality Index)
0,2025-11-23,2023,Afghanistan,AFG,2023,0.66100
1,2025-11-23,2023,Africa,NaN,2023,0.53045
2,2025-11-23,2023,Albania,ALB,2023,0.10700
3,2025-11-23,2023,Algeria,DZA,2023,0.44300
4,2025-11-23,2023,Angola,AGO,2023,0.51500


In [15]:
# Check for missing values
df.isnull().sum()

Scraped_Date                                                0
Data_Year                                                   0
Country                                                     0
ISO3                                                       20
Year                                                        0
Gender Inequality Index (GII) (Gender Inequality Index)     0
dtype: int64

In [16]:
# Data types
df.dtypes

Scraped_Date                                                object
Data_Year                                                    int64
Country                                                     object
ISO3                                                        object
Year                                                         int64
Gender Inequality Index (GII) (Gender Inequality Index)    float64
dtype: object

In [17]:
# Basic statistics
df.describe()

,Data_Year,Year,Gender Inequality Index (GII) (Gender Inequality Index)
count,193.0,193.0,193.000000
mean,2023.0,2023.0,0.331953
std,0.0,0.0,0.194076
min,2023.0,2023.0,0.003000
25%,2023.0,2023.0,0.169000
50%,2023.0,2023.0,0.352000
75%,2023.0,2023.0,0.492000
max,2023.0,2023.0,0.838000


In [18]:
# Clean data - drop rows with missing GII values
df_clean = df.dropna(subset=['Gender Inequality Index (GII) (Gender Inequality Index)'])
print('After cleaning shape:', df_clean.shape)
print('Non-null GII values:', df_clean['Gender Inequality Index (GII) (Gender Inequality Index)'].notna().sum())

After cleaning shape: (193, 6)
Non-null GII values: 193


In [19]:
# EDA: Distribution of Gender Inequality Index
fig = px.histogram(df_clean, x='Gender Inequality Index (GII) (Gender Inequality Index)', nbins=30, title='Gender Inequality Index Distribution', template='plotly_dark')
fig.update_layout(showlegend=False)
fig.write_image('../viz/gii_distribution.png')
fig.show()

In [20]:
# EDA: Top 20 countries with lowest GII (most equal)
top_equal = df_clean.nsmallest(20, 'Gender Inequality Index (GII) (Gender Inequality Index)')
fig = px.bar(top_equal, x='Country', y='Gender Inequality Index (GII) (Gender Inequality Index)', 
             title='Top 20 Most Gender Equal Countries', color='Gender Inequality Index (GII) (Gender Inequality Index)', template='plotly_dark')
fig.update_layout(xaxis_tickangle=-45)
fig.write_image('../viz/top_equal_countries.png')
fig.show()

In [21]:
# EDA: Top 20 countries with highest GII (least equal)
least_equal = df_clean.nlargest(20, 'Gender Inequality Index (GII) (Gender Inequality Index)')
fig = px.bar(least_equal, x='Country', y='Gender Inequality Index (GII) (Gender Inequality Index)', 
             title='Top 20 Least Gender Equal Countries', color='Gender Inequality Index (GII) (Gender Inequality Index)', template='plotly_dark')
fig.update_layout(xaxis_tickangle=-45)
fig.write_image('../viz/least_equal_countries.png')
fig.show()

ERROR:tornado.general:SEND Error: Host unreachable


In [22]:
# Categorize countries by inequality level
df_clean['GII_Category'] = pd.cut(df_clean['Gender Inequality Index (GII) (Gender Inequality Index)'], 
                                   bins=[0, 0.1, 0.2, 0.3, 0.5, 1.0],
                                   labels=['Very Equal', 'Equal', 'Moderate', 'Unequal', 'Very Unequal'])

# EDA: GII by Category
fig = px.pie(df_clean, names='GII_Category', title='Distribution of Gender Inequality Categories', template='plotly_dark')
fig.write_image('../viz/gii_category_distribution.png')
fig.show()

In [23]:
# EDA: Regional Analysis (if available)
regional_data = df_clean[df_clean['Country'].str.contains('Asia|Europe|Africa|America|Arab|UNDP', case=False, na=False)]
if len(regional_data) > 0:
    fig = px.bar(regional_data.groupby('Country')['Gender Inequality Index (GII) (Gender Inequality Index)'].mean().reset_index(),
                 x='Country', y='Gender Inequality Index (GII) (Gender Inequality Index)',
                 title='Average GII by Region', template='plotly_dark')
    fig.update_layout(xaxis_tickangle=-45)
    fig.write_image('../viz/gii_by_region.png')
fig.show()

In [11]:
# ML: Preprocessing
# Filter to individual countries only (exclude regional aggregates)
df_countries = df_clean[df_clean['ISO3'].notna() & (df_clean['ISO3'] != '')].copy()
print(f'Countries with ISO3 code: {len(df_countries)}')

# Encode countries
le_country = LabelEncoder()
df_countries['country_encoded'] = le_country.fit_transform(df_countries['Country'])

# Select features
X = df_countries[['country_encoded']].values
y = df_countries['Gender Inequality Index (GII) (Gender Inequality Index)'].values

print(f'Features shape: {X.shape}')
print(f'Target shape: {y.shape}')

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Countries with ISO3 code: 173
Features shape: (173, 1)
Target shape: (173,)


In [ ]:
# Train Random Forest model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)

# Predictions
y_pred = rf_model.predict(X_test_scaled)

# Evaluation
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f'MAE: {mae:.4f}')
print(f'RMSE: {rmse:.4f}')
print(f'R²: {r2:.4f}')

# Actual vs Predicted plot
fig = px.scatter(x=y_test, y=y_pred, title='Actual vs Predicted Gender Inequality Index', template='plotly_dark')
fig.add_trace(go.Scatter(x=[y_test.min(), y_test.max()], y=[y_test.min(), y_test.max()], 
                         mode='lines', name='Perfect Prediction', line=dict(dash='dash')))
fig.update_layout(xaxis_title='Actual GII', yaxis_title='Predicted GII')
fig.write_image('../viz/actual_vs_predicted.png')

MAE: 0.2352
RMSE: 0.2827
R²: -0.7132


In [13]:
# Create a pipeline for the model
from sklearn.pipeline import Pipeline
import joblib

# Define the pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

# Fit the pipeline
pipeline.fit(X_train, y_train)

# Example prediction
# Create sample data for a country (US)
sample_country = pd.DataFrame({
    'country_encoded': [le_country.transform(['United States'])[0]]
})

# Make prediction
prediction = pipeline.predict(sample_country.values)

print('Sample Gender Inequality Prediction:')
print(f'Predicted GII for United States: {prediction[0]:.4f}')
print(f'Actual GII for United States: {df_countries[df_countries["Country"] == "United States"]["Gender Inequality Index (GII) (Gender Inequality Index)"].values[0]:.4f}')

# Save the pipeline
joblib.dump(pipeline, '../models/gii_pipeline.joblib')
print('Pipeline saved to ../models/gii_pipeline.joblib')

Sample Gender Inequality Prediction:
Predicted GII for United States: 0.1777
Actual GII for United States: 0.1690
Pipeline saved to ../models/gii_pipeline.joblib
